<a href="https://colab.research.google.com/github/tmacpherson6/investment_advisor_dashboard_capstone/blob/master/Kristine_SCF_Data_Prep_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Survey of Consumer Finances (SCF) Public Data Preparation**

This notebook will prepare data from the **2022 Survey of Consumer Finances** (SCF) available here: https://www.federalreserve.gov/econres/scfindex.html. The SCF is a triennial cross-sectional survey of U.S. families.

In this notebook, we will use the **convey package** in R to prepare the data for analysis. We will utilize code from the following link as reference for downloading and converting, and constructing the complex survey design: https://www.convey-r.org/1.6-survey-of-consumer-finances-scf.html#survey-of-consumer-finances-scf




### **Packages**

In [2]:
# Comment out if libraries below have already been installed
install.packages("haven")
install.packages("stringr")
install.packages("survey")
install.packages("mitools")
install.packages("convey")
install.packages("scf")
install.packages("httr")
install.packages("dplyr")

library(haven)
library(stringr)
library(survey)
library(mitools)
library(convey)
library(scf)
library(httr)
library(dplyr)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Warning message in install.packages("stringr"):
“installation of package ‘stringr’ had non-zero exit status”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Loading required package: grid

Loading required package: Matrix

Loading required package: survival


Attaching package: ‘survey’


The following object is masked from ‘package:graphics’:

    dotchart



Attaching package: ‘s

### **Download the data**
Define a function to download and import each STATA file:

In [3]:
scf_dta_import <-
  function(this_url) {
    this_tf <- tempfile()
    download.file(this_url , this_tf , mode = 'wb')
    this_tbl <- read_dta(this_tf)
    this_df <- data.frame(this_tbl)
    file.remove(this_tf)
    names(this_df) <- tolower(names(this_df))
    this_df
  }

Download and import the full, summary extract, and replicate weights tables

In [4]:
scf_df <-
  scf_dta_import("https://www.federalreserve.gov/econres/files/scf2022s.zip")

ext_df <-
  scf_dta_import("https://www.federalreserve.gov/econres/files/scfp2022s.zip")

scf_rw_df <-
  scf_dta_import("https://www.federalreserve.gov/econres/files/scf2022rw1s.zip")

Confirm both the full public data and the summary extract contain five records per family:

In [5]:
stopifnot(nrow(scf_df) == nrow(scf_rw_df) * 5)
stopifnot(nrow(scf_df) == nrow(ext_df))

Confirm only the primary economic unit and the five implicate identifiers overlap:

In [6]:
stopifnot(all(sort(intersect(
  names(scf_df) , names(ext_df)
)) == c('y1' , 'yy1')))

stopifnot(all(sort(intersect(
  names(scf_df) , names(scf_rw_df)
)) == c('y1' , 'yy1')))

stopifnot(all(sort(intersect(
  names(ext_df) , names(scf_rw_df)
)) == c('y1' , 'yy1')))

### **Calculate replicate weights**

Remove the implicate identifier from the replicate weights table, add a column of fives for weighting:

In [7]:
scf_rw_df[, 'y1'] <- NULL

scf_df[, 'five'] <- 5

Break the main table into five different implicates based on the final character of the column y1:

In [8]:
s1_df <- scf_df[str_sub(scf_df[, 'y1'] ,-1 ,-1) == 1 ,]
s2_df <- scf_df[str_sub(scf_df[, 'y1'] ,-1 ,-1) == 2 ,]
s3_df <- scf_df[str_sub(scf_df[, 'y1'] ,-1 ,-1) == 3 ,]
s4_df <- scf_df[str_sub(scf_df[, 'y1'] ,-1 ,-1) == 4 ,]
s5_df <- scf_df[str_sub(scf_df[, 'y1'] ,-1 ,-1) == 5 ,]

Combine these into a single list, then merge each implicate with the summary extract:

In [9]:
scf_imp <- list(s1_df , s2_df , s3_df , s4_df , s5_df)

scf_list <- lapply(scf_imp , merge , ext_df)

Replace all missing values in the replicate weights table with zeroes, multiply the replicate weights by the multiplication factor, then only keep the unique identifier and the final (combined) replicate weights:

In [10]:
scf_rw_df[is.na(scf_rw_df)] <- 0

scf_rw_df[, paste0('wgt' , 1:999)] <-
  scf_rw_df[, paste0('wt1b' , 1:999)] * scf_rw_df[, paste0('mm' , 1:999)]

scf_rw_df <- scf_rw_df[, c('yy1' , paste0('wgt' , 1:999))]

In [16]:
head(scf_rw_df,10)

,yy1,wgt1,wgt2,wgt3,wgt4,wgt5,wgt6,wgt7,wgt8,wgt9,⋯,wgt990,wgt991,wgt992,wgt993,wgt994,wgt995,wgt996,wgt997,wgt998,wgt999
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,1,26451.2230,0.0000,14002.6729,0.0000,0.000,16874.5127,0.0000,18741.8782,15289.793,⋯,0.0000,49965.9262,29117.157,0.000,0.00,14943.4463,0.000,19117.858,16439.5694,0.0000
2,2,6289.0566,1411.6469,983.2385,0.0000,3084.613,1154.2066,1799.5033,1411.9840,1192.578,⋯,1300.6444,884.1410,2465.763,1236.539,0.00,1429.6358,1452.847,0.000,1062.0580,4617.4067
3,3,6774.8866,8037.1922,7258.5912,3858.4697,0.000,7551.7450,3888.4944,6456.9769,0.000,⋯,4153.7145,7747.0939,2970.313,9561.832,0.00,0.0000,11370.593,4149.356,0.0000,3332.8323
4,4,0.0000,9936.2161,6223.4740,4282.0886,7766.534,5332.4129,5201.8238,2934.9077,0.000,⋯,0.0000,2533.3639,0.000,4578.931,11861.15,10112.4030,6856.440,0.000,6227.3983,5677.8928
5,5,0.0000,40619.3407,63649.6889,42979.0259,46699.580,35679.0895,33638.8711,32748.5001,70194.180,⋯,69083.1096,27183.7433,41378.908,53687.596,0.00,0.0000,69310.726,84768.029,34920.9256,29940.4786
6,6,0.0000,24479.9818,0.0000,15545.5333,17312.882,0.0000,32282.1777,0.0000,0.000,⋯,47362.7819,17069.0121,0.000,29337.313,17926.13,9580.2134,45809.555,0.000,19979.6312,22572.3263
7,7,367.2689,469.7428,930.8558,305.7598,0.000,262.6821,533.9634,489.7802,317.984,⋯,489.8603,712.4673,0.000,0.000,0.00,297.3545,0.000,0.000,326.9593,592.7694
8,8,0.0000,23780.8355,12895.4796,24056.6972,13066.478,13193.1899,11986.2542,0.0000,12181.494,⋯,22443.7856,11835.4952,22836.831,0.000,11485.04,0.0000,11846.010,12733.323,22376.3603,12266.6661
9,9,24950.8287,0.0000,17135.4664,49285.4121,0.000,48575.6292,0.0000,0.0000,0.000,⋯,35863.4513,39719.4416,0.000,0.000,14441.54,0.0000,0.000,30226.482,15459.3784,14968.5372


Sort both the five implicates and also the replicate weights table by the unique identifier:

In [11]:
scf_list <-
  lapply(scf_list , function(w)
    w[order(w[, 'yy1']) ,])

scf_rw_df <- scf_rw_df[order(scf_rw_df[, 'yy1']) ,]

Save cleaned data as a csv. If you are using Google Colab, you can download the files by clicking the file icon on the left-hand side of the screen.

"scf_list.csv" contains the full dataset including all implicates. Implicates can be filtered out based on the final character in column y1.

"scf_rw_df" contains the replicate weights for each record. The file contains the case ID in yy1, along with 999 replicate-weight columns, one for each of the 999 bootstrap replicate samples used to estimate sampling variability.

More information about the replicate weights and imputations are available here: https://www.federalreserve.gov/econres/files/Standard_Error_Documentation.pdf

In [13]:
# SCF Variables
write.csv(bind_rows(scf_list), "scf_list.csv", row.names = FALSE)

# Replicate weights table
write.csv(scf_rw_df, "scf_rw_df.csv", row.names = FALSE)